In [ ]:
# Title: 03_Astre_PCR.ipynb
# Author: Lajoyce Mboning
# Date:2025
# Related publication: Emma K. Costa, and Jingxun Chen, in prep


# Description - Run the PCR clock on Astre et al

# Set-Up

In [1]:
import os
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from scipy.stats import poisson
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Lasso
from numpy import arange
from numpy import absolute
from numpy import mean
from numpy import std
import numpy as np
from scipy import stats
from sklearn.metrics import mean_absolute_error
import statsmodels.api as sm
lowess = sm.nonparametric.lowess
from matplotlib.lines import Line2D
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, make_scorer
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from scipy.stats import ttest_ind, mannwhitneyu
import random
import itertools

In [49]:
os.chdir("/labs/twc/Emma/atlas_gdrive_backup/revisions/CodeCheck_revisions/Astre_model_outputs/PCR/")

In [3]:
palette = sns.color_palette("deep")
hex_codes = palette.as_hex()

print(hex_codes)

['#4c72b0', '#dd8452', '#55a868', '#c44e52', '#8172b3', '#937860', '#da8bc3', '#8c8c8c', '#ccb974', '#64b5cd']


In [4]:
# List of directories you want to create
dirs_to_create = [
    "./reference/",
    "./predictions/",
    "./gene_sets/",
    "./plots/",
    "./predictions_exhaustive/",
    "./reference_exhaustive/"
]


for d in dirs_to_create:
    os.makedirs(d, exist_ok=True)  # exist_ok=True won't raise an error if it already exists

### Load data

In [5]:
df_norm = pd.read_csv("../../AtlasFiles_forLajoyce_240507/CountsNormDESeq2_AllTissue_240506.csv")

In [6]:
df_norm = df_norm.T

In [7]:
df_norm.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A1_1,23876.707871,687.004459,2181.018965,576.907591,221.294706,884.077854,1368.504075,1348.686639,619.845370,636.359900,...,913.804009,178.356927,1117.483215,243.314079,1718.612117,47.341653,568.099841,3323.824460,1566.678439,358.915791
A1_2,36.845369,407.384647,349.683408,374.015256,493.588906,469.257059,963.541161,1619.805848,453.962755,449.791581,...,774.447946,64.653195,670.168600,107.060129,777.923924,19.465478,672.254187,2691.797529,191.873997,253.746410
A10_1,4821.965545,907.596706,218.510659,554.051370,191.503723,684.175695,1125.288972,662.897504,586.787050,637.527352,...,520.497299,128.487541,1351.165159,112.938093,1122.015404,311.807344,542.593883,2612.307200,408.377598,391.191366
A10_2,4.382338,451.380822,1198.256441,303.633424,289.860362,445.120339,1093.080327,468.284126,709.312721,492.073962,...,1201.386683,131.470142,800.089724,174.667475,1304.058604,28.172173,1264.617561,1144.416288,157.764171,238.524401
A11_1,7011.583272,766.674115,222.103168,627.278822,272.285474,675.602523,1267.567871,975.767056,739.724358,713.703904,...,420.973787,192.365505,1129.101879,109.657631,1059.404232,398.670540,521.338398,1948.746206,360.569160,446.064940


In [8]:
df_norm_frequency = df_norm.div(df_norm.sum(axis=1), axis=0)

In [9]:
df_metadata = pd.read_csv("../../AtlasFiles_forLajoyce_240507/ExperimentDesign_allbatches_combined_v7.csv")

df_metadata.head(3)

,Unnamed: 0,animalID,sex,cohort,age_days,harvest_date,hatch_date,tissue,tissue_grind_date,RNA_extract_date,RNA_batch,RNA_extractor,RNAID,cDNA_batch,plate_well,sampleNames,lib,censored,censor_code,notes
0,A1_1,J9,F,2,155,8/16/22,3/14/22,Gut,NaN,3/8/23,Gut_1,EC,RNA352,1,A1,A1,A1_1,NaN,NaN,NaN
1,A1_2,A01,M,2,78,5/31/22,3/14/22,Bone,4/19/23,4/22/23,Bone_1,EC,RNA572,2,A1,A1,A1_2,NaN,NaN,NaN
2,A10_1,P_1B_10,M,1B,133,1/31/22,9/20/21,Kidney,NaN,3/4/23,Kidney_1,JC,R258,1,A10,A10,A10_1,NaN,NaN,NaN


In [10]:
df_metadata.rename(columns={"Unnamed: 0": "sample"}, inplace=True)

df_metadata.head(3)

,sample,animalID,sex,cohort,age_days,harvest_date,hatch_date,tissue,tissue_grind_date,RNA_extract_date,RNA_batch,RNA_extractor,RNAID,cDNA_batch,plate_well,sampleNames,lib,censored,censor_code,notes
0,A1_1,J9,F,2,155,8/16/22,3/14/22,Gut,NaN,3/8/23,Gut_1,EC,RNA352,1,A1,A1,A1_1,NaN,NaN,NaN
1,A1_2,A01,M,2,78,5/31/22,3/14/22,Bone,4/19/23,4/22/23,Bone_1,EC,RNA572,2,A1,A1,A1_2,NaN,NaN,NaN
2,A10_1,P_1B_10,M,1B,133,1/31/22,9/20/21,Kidney,NaN,3/4/23,Kidney_1,JC,R258,1,A10,A10,A10_1,NaN,NaN,NaN


In [11]:
df_metadata.set_index("sample", inplace=True)

In [12]:
# Merge based on index
combined_df = pd.concat([df_norm_frequency, df_metadata[['sex', 'tissue', 'age_days']]], axis=1)

In [13]:
combined_df["tissue"].value_counts()

Gut           55
Kidney        55
Muscle        55
Spleen        55
SpinalCord    54
Heart         54
Skin          54
Brain         54
Liver         54
Fat           53
Bone          51
Eye           37
Testis        31
Ovary         15
Name: tissue, dtype: int64

In [14]:
df_norm_metadata = pd.concat([df_norm, df_metadata[["sex", "tissue", "age_days"]]], axis=1)

df_norm_metadata.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A1_1,23876.707871,687.004459,2181.018965,576.907591,221.294706,884.077854,1368.504075,1348.686639,619.845370,636.359900,...,243.314079,1718.612117,47.341653,568.099841,3323.824460,1566.678439,358.915791,F,Gut,155
A1_2,36.845369,407.384647,349.683408,374.015256,493.588906,469.257059,963.541161,1619.805848,453.962755,449.791581,...,107.060129,777.923924,19.465478,672.254187,2691.797529,191.873997,253.746410,M,Bone,78
A10_1,4821.965545,907.596706,218.510659,554.051370,191.503723,684.175695,1125.288972,662.897504,586.787050,637.527352,...,112.938093,1122.015404,311.807344,542.593883,2612.307200,408.377598,391.191366,M,Kidney,133
A10_2,4.382338,451.380822,1198.256441,303.633424,289.860362,445.120339,1093.080327,468.284126,709.312721,492.073962,...,174.667475,1304.058604,28.172173,1264.617561,1144.416288,157.764171,238.524401,M,SpinalCord,75
A11_1,7011.583272,766.674115,222.103168,627.278822,272.285474,675.602523,1267.567871,975.767056,739.724358,713.703904,...,109.657631,1059.404232,398.670540,521.338398,1948.746206,360.569160,446.064940,M,Kidney,47


In [15]:
#function to load norm counts
def load_intervention_countdata(tissue):
    file_path = f"../../Data_from_Others/Astre_et_al_2023/remap_EKC/CountsNormDESeq2_AstreLiver_250605.csv"
    return pd.read_csv(file_path)

#function to load metadata
def load_intervention_metadata(tissue):
    file_path = f"../../Data_from_Others/Astre_et_al_2023/remap_EKC/ExperimentDesign_AstreLiver_update_20250629.csv"
    return pd.read_csv(file_path)

In [16]:
select_tissue = 'Liver'

In [17]:
#load intervention data
df_intervention = load_intervention_countdata(select_tissue)
df_intervention.head(5)

,Unnamed: 0,SRR17215646,SRR17215657,SRR17215658,SRR17215659,SRR17215660,SRR17215661,SRR17215662,SRR17215663,SRR17215664,...,SRR17215652,SRR17215653,SRR17215654,SRR17215655,SRR17215656,SRR17215674,SRR17215675,SRR17215676,SRR17215682,SRR17215685
0,LOC107380275,3.415293,0.000000,1.014282,4.449614,5.352522,5.628138,1.938696,2.401523,3.746958,...,0.000000,0.918583,2.497475,3.017397,0.910767,2.739876,4.052651,0.000000,3.892290,3.967226
1,LOC107389369,0.000000,0.000000,0.000000,0.000000,0.000000,0.938023,0.969348,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.350884,0.000000,0.000000,0.000000
2,klf12,367.713240,485.228400,454.398481,422.713353,611.376908,321.741915,648.493802,481.905627,445.887995,...,632.040153,378.456315,371.291324,316.826734,348.823867,269.877814,617.353883,369.988177,356.144533,247.290424
3,enpp2,9.107449,18.835113,9.128541,21.135668,13.083942,17.822438,20.356308,16.010154,11.240874,...,30.097150,27.557499,16.649835,13.578289,8.196905,38.358268,9.456186,5.736251,29.192175,18.513722
4,LOC107382607,33.583717,40.360957,54.771245,54.507774,23.194260,80.669984,55.252835,45.628938,43.464712,...,36.604642,59.707914,43.289571,48.278360,41.895295,90.415917,63.491537,97.516264,66.168930,85.956565


In [18]:
#load intervention metadata
df_intervention_metadata = load_intervention_metadata(select_tissue)
df_intervention_metadata.set_index("Run", inplace=True)
print(df_intervention_metadata.head(1))

                     lib       age Assay.Type  AvgSpotLen       Bases  \
Run                                                                     
SRR17215646  SRR17215646  15 weeks    RNA-Seq          74  3525629281   

              BioProject     BioSample       Bytes Center.Name Consent  ...  \
Run                                                                     ...   
SRR17215646  PRJNA788399  SAMN23970501  1614398132         GEO  public  ...   

                      create_date version Sample.Name   sex source_name  \
Run                                                                       
SRR17215646  2021-12-13T12:06:00Z       1  GSM5730981  male   Nfr Liver   

             SRA.Study tissue                       file UsedForTraining  \
Run                                                                        
SRR17215646  SRP350511  Liver  SRR17215646.featureCounts             Yes   

            ExcludedFromAnalysis  
Run                               
SRR17215646      

### Reformat Data

In [19]:
#reformat intervention dataset
df_intervention.rename(columns={"Unnamed: 0": "Run"}, inplace=True)
df_intervention.set_index("Run", inplace=True)
df_intervention = df_intervention.T
df_intervention.head(3)

Run,LOC107380275,LOC107389369,klf12,enpp2,LOC107382607,LOC107386405,wwox,LOC107388920,csgr14h3orf70,LOC107376005,...,LOC107391152,LOC107394127,golph3,LOC107381980,LOC107390548,lhfpl5,LOC107373016,LOC107392495,LOC107396697,LOC107391667
SRR17215646,3.415293,0.0,367.713240,9.107449,33.583717,119.535264,80.828607,56.352339,72.290374,169.057016,...,1.707647,0.0,319.899134,1.138431,135.473299,0.0,0.569216,134.904083,0.0,0.0
SRR17215657,0.000000,0.0,485.228400,18.835113,40.360957,45.742418,43.051688,37.670227,47.536239,258.310128,...,0.896910,0.0,254.722487,0.000000,44.845508,0.0,0.000000,163.237650,0.0,0.0
SRR17215658,1.014282,0.0,454.398481,9.128541,54.771245,43.614140,50.714116,55.785528,51.728398,278.927639,...,2.028565,0.0,269.799098,0.000000,29.414187,0.0,0.000000,146.056655,0.0,0.0


In [20]:
df_intervention_age = df_intervention.copy()

In [21]:
df_intervention_metadata = df_intervention_metadata[df_intervention_metadata["tissue"] == select_tissue]
df_intervention_metadata.head(5)

,lib,age,Assay.Type,AvgSpotLen,Bases,BioProject,BioSample,Bytes,Center.Name,Consent,...,create_date,version,Sample.Name,sex,source_name,SRA.Study,tissue,file,UsedForTraining,ExcludedFromAnalysis
Run,,,,,,,,,,,,,,,,,,,,,
SRR17215646,SRR17215646,15 weeks,RNA-Seq,74,3525629281,PRJNA788399,SAMN23970501,1614398132,GEO,public,...,2021-12-13T12:06:00Z,1,GSM5730981,male,Nfr Liver,SRP350511,Liver,SRR17215646.featureCounts,Yes,Include
SRR17215657,SRR17215657,6.5 weeks,RNA-Seq,74,2840020025,PRJNA788399,SAMN23970512,1302718445,GEO,public,...,2021-12-13T12:07:00Z,1,GSM5730992,male,Nfr Liver,SRP350511,Liver,SRR17215657.featureCounts,No,Include
SRR17215658,SRR17215658,6.5 weeks,RNA-Seq,74,2919651799,PRJNA788399,SAMN23970491,1345051064,GEO,public,...,2021-12-13T12:07:00Z,1,GSM5730993,male,Nfr Liver,SRP350511,Liver,SRR17215658.featureCounts,No,Include
SRR17215659,SRR17215659,6.5 weeks,RNA-Seq,74,2295530551,PRJNA788399,SAMN23970492,1061958088,GEO,public,...,2021-12-13T12:04:00Z,1,GSM5730994,male,Nfr Liver,SRP350511,Liver,SRR17215659.featureCounts,No,Include
SRR17215660,SRR17215660,15 weeks,RNA-Seq,74,3363520811,PRJNA788399,SAMN23970493,1511631192,GEO,public,...,2021-12-13T12:07:00Z,1,GSM5730995,male,Nfr Liver,SRP350511,Liver,SRR17215660.featureCounts,NaN,Exclude


In [22]:
df_intervention_metadata = df_intervention_metadata[df_intervention_metadata["sex"] == 'male']
df_intervention_metadata.head(5)

,lib,age,Assay.Type,AvgSpotLen,Bases,BioProject,BioSample,Bytes,Center.Name,Consent,...,create_date,version,Sample.Name,sex,source_name,SRA.Study,tissue,file,UsedForTraining,ExcludedFromAnalysis
Run,,,,,,,,,,,,,,,,,,,,,
SRR17215646,SRR17215646,15 weeks,RNA-Seq,74,3525629281,PRJNA788399,SAMN23970501,1614398132,GEO,public,...,2021-12-13T12:06:00Z,1,GSM5730981,male,Nfr Liver,SRP350511,Liver,SRR17215646.featureCounts,Yes,Include
SRR17215657,SRR17215657,6.5 weeks,RNA-Seq,74,2840020025,PRJNA788399,SAMN23970512,1302718445,GEO,public,...,2021-12-13T12:07:00Z,1,GSM5730992,male,Nfr Liver,SRP350511,Liver,SRR17215657.featureCounts,No,Include
SRR17215658,SRR17215658,6.5 weeks,RNA-Seq,74,2919651799,PRJNA788399,SAMN23970491,1345051064,GEO,public,...,2021-12-13T12:07:00Z,1,GSM5730993,male,Nfr Liver,SRP350511,Liver,SRR17215658.featureCounts,No,Include
SRR17215659,SRR17215659,6.5 weeks,RNA-Seq,74,2295530551,PRJNA788399,SAMN23970492,1061958088,GEO,public,...,2021-12-13T12:04:00Z,1,GSM5730994,male,Nfr Liver,SRP350511,Liver,SRR17215659.featureCounts,No,Include
SRR17215660,SRR17215660,15 weeks,RNA-Seq,74,3363520811,PRJNA788399,SAMN23970493,1511631192,GEO,public,...,2021-12-13T12:07:00Z,1,GSM5730995,male,Nfr Liver,SRP350511,Liver,SRR17215660.featureCounts,NaN,Exclude


In [23]:
#subset intervention counts to your samples of interest
df_intervention_age = df_intervention_age.loc[df_intervention_metadata.index.tolist(),]
df_intervention_age.head(5)

Run,LOC107380275,LOC107389369,klf12,enpp2,LOC107382607,LOC107386405,wwox,LOC107388920,csgr14h3orf70,LOC107376005,...,LOC107391152,LOC107394127,golph3,LOC107381980,LOC107390548,lhfpl5,LOC107373016,LOC107392495,LOC107396697,LOC107391667
SRR17215646,3.415293,0.0,367.713240,9.107449,33.583717,119.535264,80.828607,56.352339,72.290374,169.057016,...,1.707647,0.0,319.899134,1.138431,135.473299,0.0,0.569216,134.904083,0.0,0.0
SRR17215657,0.000000,0.0,485.228400,18.835113,40.360957,45.742418,43.051688,37.670227,47.536239,258.310128,...,0.896910,0.0,254.722487,0.000000,44.845508,0.0,0.000000,163.237650,0.0,0.0
SRR17215658,1.014282,0.0,454.398481,9.128541,54.771245,43.614140,50.714116,55.785528,51.728398,278.927639,...,2.028565,0.0,269.799098,0.000000,29.414187,0.0,0.000000,146.056655,0.0,0.0
SRR17215659,4.449614,0.0,422.713353,21.135668,54.507774,58.957389,54.507774,63.407003,87.879881,273.651276,...,0.000000,0.0,230.267537,0.000000,23.360475,0.0,0.000000,151.286884,0.0,0.0
SRR17215660,5.352522,0.0,611.376908,13.083942,23.194260,55.309390,61.851360,67.798607,79.693099,228.968978,...,0.000000,0.0,234.916225,1.784174,170.091241,0.0,0.000000,150.465329,0.0,0.0


In [24]:
#select tissue and sex from atlas dataset
tissue_df_norm = df_norm_metadata.loc[df_norm_metadata.iloc[:,-2] == select_tissue]
tissue_df_norm = tissue_df_norm.loc[tissue_df_norm.iloc[:,-3] == 'M']
tissue_df_norm.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A5_1,100443.541866,569.498293,9197.259866,817.106246,41.267992,1034.451005,982.178215,4385.411973,935.407824,558.493495,...,101.794381,1141.747785,2.751199,511.723104,2555.864318,299.880743,448.445515,M,Liver,134
A8_1,139272.752006,589.285647,3047.097867,1149.107012,76.116063,1023.883812,1092.633804,2536.383640,989.508816,643.303498,...,130.133914,1156.473083,22.098212,564.732079,2492.187216,378.124957,414.955310,M,Liver,133
B7_1,106880.160432,442.985714,13625.869465,670.276813,71.898205,1080.792369,1289.529093,2620.805531,772.325878,712.024158,...,34.789454,1338.234329,6.957891,547.354076,2435.261777,582.143529,565.908451,M,Liver,133
C5_1,113772.810228,696.735509,24082.457958,633.892699,62.842811,1158.493553,920.783791,3191.321862,857.940980,740.452247,...,166.670063,1155.761257,24.590665,478.151820,2330.648586,237.709762,437.167378,M,Liver,162
C7_1,109847.738206,626.472460,8772.952023,818.154332,91.165768,1302.034180,1491.378468,2718.609966,885.944262,883.606679,...,179.993953,766.727488,16.363087,523.618773,2966.393850,367.000658,446.478507,M,Liver,152


In [25]:
tissue_df_age = tissue_df_norm.drop(tissue_df_norm.columns[[-2, -3]], axis=1)
tissue_df_age.head(2)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age_days
A5_1,100443.541866,569.498293,9197.259866,817.106246,41.267992,1034.451005,982.178215,4385.411973,935.407824,558.493495,...,71.531187,723.565464,101.794381,1141.747785,2.751199,511.723104,2555.864318,299.880743,448.445515,134
A8_1,139272.752006,589.285647,3047.097867,1149.107012,76.116063,1023.883812,1092.633804,2536.383640,989.508816,643.303498,...,56.473208,785.714196,130.133914,1156.473083,22.098212,564.732079,2492.187216,378.124957,414.955310,133


In [26]:
df_intervention_metadata.columns

Index(['lib', 'age', 'Assay.Type', 'AvgSpotLen', 'Bases', 'BioProject',
       'BioSample', 'Bytes', 'Center.Name', 'Consent', 'DATASTORE.filetype',
       'DATASTORE.provider', 'DATASTORE.region', 'Experiment',
       'feeding_condition', 'genotype', 'GEO_Accession..exp.', 'Instrument',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample.Name',
       'sex', 'source_name', 'SRA.Study', 'tissue', 'file', 'UsedForTraining',
       'ExcludedFromAnalysis'],
      dtype='object')

In [27]:
df_intervention_age["tissue"] = df_intervention_metadata["tissue"].tolist()
df_intervention_age["age"] = df_intervention_metadata["age"].tolist()
df_intervention_age["sex"] = df_intervention_metadata["sex"].tolist()
df_intervention_age["training"] = df_intervention_metadata["UsedForTraining"].tolist()
df_intervention_age["genotype"] = df_intervention_metadata["genotype"].tolist()
df_intervention_age["feeding_condition"] = df_intervention_metadata["feeding_condition"].tolist()


age_mapping = {
    "6.5 weeks": 46, #round up from 45.5
    "15 weeks": 105
}

df_intervention_age["age"] = df_intervention_age["age"].map(age_mapping)

df_intervention_age.head(3)

Run,LOC107380275,LOC107389369,klf12,enpp2,LOC107382607,LOC107386405,wwox,LOC107388920,csgr14h3orf70,LOC107376005,...,LOC107373016,LOC107392495,LOC107396697,LOC107391667,tissue,age,sex,training,genotype,feeding_condition
SRR17215646,3.415293,0.0,367.713240,9.107449,33.583717,119.535264,80.828607,56.352339,72.290374,169.057016,...,0.569216,134.904083,0.0,0.0,Liver,105,male,Yes,WT,full
SRR17215657,0.000000,0.0,485.228400,18.835113,40.360957,45.742418,43.051688,37.670227,47.536239,258.310128,...,0.000000,163.237650,0.0,0.0,Liver,46,male,No,Het,full
SRR17215658,1.014282,0.0,454.398481,9.128541,54.771245,43.614140,50.714116,55.785528,51.728398,278.927639,...,0.000000,146.056655,0.0,0.0,Liver,46,male,No,Het,full


In [28]:
df_intervention_age_counts = df_intervention_age.iloc[:,:-6]

df_intervention_age_counts.head(2)

Run,LOC107380275,LOC107389369,klf12,enpp2,LOC107382607,LOC107386405,wwox,LOC107388920,csgr14h3orf70,LOC107376005,...,LOC107391152,LOC107394127,golph3,LOC107381980,LOC107390548,lhfpl5,LOC107373016,LOC107392495,LOC107396697,LOC107391667
SRR17215646,3.415293,0.0,367.71324,9.107449,33.583717,119.535264,80.828607,56.352339,72.290374,169.057016,...,1.707647,0.0,319.899134,1.138431,135.473299,0.0,0.569216,134.904083,0.0,0.0
SRR17215657,0.000000,0.0,485.22840,18.835113,40.360957,45.742418,43.051688,37.670227,47.536239,258.310128,...,0.896910,0.0,254.722487,0.000000,44.845508,0.0,0.000000,163.237650,0.0,0.0


# Sex split clock - use SOME - WT, fed for calibration

In [29]:
#load atlas
tissue_df_age = tissue_df_age.rename(columns={'age_days': 'age'})
tissue_df_age.head(2)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A5_1,100443.541866,569.498293,9197.259866,817.106246,41.267992,1034.451005,982.178215,4385.411973,935.407824,558.493495,...,71.531187,723.565464,101.794381,1141.747785,2.751199,511.723104,2555.864318,299.880743,448.445515,134
A8_1,139272.752006,589.285647,3047.097867,1149.107012,76.116063,1023.883812,1092.633804,2536.383640,989.508816,643.303498,...,56.473208,785.714196,130.133914,1156.473083,22.098212,564.732079,2492.187216,378.124957,414.955310,133


In [30]:
#load intervention
df_intervention_age_only = df_intervention_age.drop(df_intervention_age.columns[[-1,-2,-3, -4, -6]], axis=1)
df_intervention_age_only.head(2)

Run,LOC107380275,LOC107389369,klf12,enpp2,LOC107382607,LOC107386405,wwox,LOC107388920,csgr14h3orf70,LOC107376005,...,LOC107394127,golph3,LOC107381980,LOC107390548,lhfpl5,LOC107373016,LOC107392495,LOC107396697,LOC107391667,age
SRR17215646,3.415293,0.0,367.71324,9.107449,33.583717,119.535264,80.828607,56.352339,72.290374,169.057016,...,0.0,319.899134,1.138431,135.473299,0.0,0.569216,134.904083,0.0,0.0,105
SRR17215657,0.000000,0.0,485.22840,18.835113,40.360957,45.742418,43.051688,37.670227,47.536239,258.310128,...,0.0,254.722487,0.000000,44.845508,0.0,0.000000,163.237650,0.0,0.0,46


In [31]:
#load atlas
tissue_df_age.head(2)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A5_1,100443.541866,569.498293,9197.259866,817.106246,41.267992,1034.451005,982.178215,4385.411973,935.407824,558.493495,...,71.531187,723.565464,101.794381,1141.747785,2.751199,511.723104,2555.864318,299.880743,448.445515,134
A8_1,139272.752006,589.285647,3047.097867,1149.107012,76.116063,1023.883812,1092.633804,2536.383640,989.508816,643.303498,...,56.473208,785.714196,130.133914,1156.473083,22.098212,564.732079,2492.187216,378.124957,414.955310,133


In [32]:
#load intervention
df_intervention_age_only.head(2)

Run,LOC107380275,LOC107389369,klf12,enpp2,LOC107382607,LOC107386405,wwox,LOC107388920,csgr14h3orf70,LOC107376005,...,LOC107394127,golph3,LOC107381980,LOC107390548,lhfpl5,LOC107373016,LOC107392495,LOC107396697,LOC107391667,age
SRR17215646,3.415293,0.0,367.71324,9.107449,33.583717,119.535264,80.828607,56.352339,72.290374,169.057016,...,0.0,319.899134,1.138431,135.473299,0.0,0.569216,134.904083,0.0,0.0,105
SRR17215657,0.000000,0.0,485.22840,18.835113,40.360957,45.742418,43.051688,37.670227,47.536239,258.310128,...,0.0,254.722487,0.000000,44.845508,0.0,0.000000,163.237650,0.0,0.0,46


In [33]:
df_intervention_age.head(2)

Run,LOC107380275,LOC107389369,klf12,enpp2,LOC107382607,LOC107386405,wwox,LOC107388920,csgr14h3orf70,LOC107376005,...,LOC107373016,LOC107392495,LOC107396697,LOC107391667,tissue,age,sex,training,genotype,feeding_condition
SRR17215646,3.415293,0.0,367.71324,9.107449,33.583717,119.535264,80.828607,56.352339,72.290374,169.057016,...,0.569216,134.904083,0.0,0.0,Liver,105,male,Yes,WT,full
SRR17215657,0.000000,0.0,485.22840,18.835113,40.360957,45.742418,43.051688,37.670227,47.536239,258.310128,...,0.000000,163.237650,0.0,0.0,Liver,46,male,No,Het,full


In [34]:
#for calibration, only use WT full samples
WT_samples = df_intervention_age[
    (df_intervention_age['genotype'] == 'WT') &
    (df_intervention_age['feeding_condition'] == 'full')].index

WT_samples = sorted(WT_samples)
WT_samples

['SRR17215646',
 'SRR17215647',
 'SRR17215648',
 'SRR17215653',
 'SRR17215654',
 'SRR17215655']

In [35]:
random.seed(123)

# Separate the samples
young_samples = [s for s in WT_samples if s in df_intervention_age.index and df_intervention_age.loc[s, 'age'] == 46]
old_samples = [s for s in WT_samples if s in df_intervention_age.index and df_intervention_age.loc[s, 'age'] == 105]

# Randomly sample 2 from each
random_young = random.sample(young_samples, 2)
random_old = random.sample(old_samples, 2)

# Combine the result
random_subset = random_young + random_old

#select your training samples
select_WT_samples = random_subset

df_intervention_WT = df_intervention_age.loc[
    df_intervention_age.index.isin(select_WT_samples)
]

df_intervention_WT

Run,LOC107380275,LOC107389369,klf12,enpp2,LOC107382607,LOC107386405,wwox,LOC107388920,csgr14h3orf70,LOC107376005,...,LOC107373016,LOC107392495,LOC107396697,LOC107391667,tissue,age,sex,training,genotype,feeding_condition
SRR17215646,3.415293,0.0,367.713240,9.107449,33.583717,119.535264,80.828607,56.352339,72.290374,169.057016,...,0.569216,134.904083,0.0,0.0,Liver,105,male,Yes,WT,full
SRR17215647,3.145674,0.0,286.885472,30.827606,63.542615,62.913481,72.979638,76.125312,18.874044,121.423018,...,0.000000,145.330140,0.0,0.0,Liver,105,male,Yes,WT,full
SRR17215653,0.918583,0.0,378.456315,27.557499,59.707914,59.707914,56.952164,76.242413,33.068998,214.948490,...,0.000000,133.194577,0.0,0.0,Liver,46,male,Yes,WT,full
SRR17215654,2.497475,0.0,371.291324,16.649835,43.289571,54.944456,48.284522,47.452030,49.117014,269.727329,...,0.000000,145.686058,0.0,0.0,Liver,46,male,Yes,WT,full


In [36]:
df_intervention_WT = df_intervention_WT.drop(df_intervention_WT.columns[[-1,-2,-3, -4, -6]], axis=1)
df_intervention_WT.head(2)

Run,LOC107380275,LOC107389369,klf12,enpp2,LOC107382607,LOC107386405,wwox,LOC107388920,csgr14h3orf70,LOC107376005,...,LOC107394127,golph3,LOC107381980,LOC107390548,lhfpl5,LOC107373016,LOC107392495,LOC107396697,LOC107391667,age
SRR17215646,3.415293,0.0,367.713240,9.107449,33.583717,119.535264,80.828607,56.352339,72.290374,169.057016,...,0.0,319.899134,1.138431,135.473299,0.0,0.569216,134.904083,0.0,0.0,105
SRR17215647,3.145674,0.0,286.885472,30.827606,63.542615,62.913481,72.979638,76.125312,18.874044,121.423018,...,0.0,378.739154,0.000000,114.502535,0.0,0.000000,145.330140,0.0,0.0,105


In [37]:
#combine the dataframes
atlas_plus_intervention = pd.concat([tissue_df_age, df_intervention_WT], axis=0)

#atlas_plus_intervention.head(3)
atlas_plus_intervention.tail(3)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
SRR17215647,37283.786912,235.296418,2578.823572,295.693359,24.536257,287.514607,296.322494,340.361930,346.653278,149.734084,...,10.066157,961.317985,38.377223,523.440159,162.316780,253.541327,1032.410218,800.888609,295.064224,105
SRR17215653,36187.588695,135.031744,137.787493,230.564406,33.987582,208.518407,317.829818,471.233228,188.309574,114.822911,...,9.185833,716.494966,44.091998,281.086487,0.918583,273.737820,1640.589755,551.149974,329.771401,46
SRR17215654,37290.635782,127.371239,159.838417,269.727329,13.319868,164.000876,311.351917,482.845219,178.153236,108.223928,...,9.157409,699.293076,42.457080,207.290448,3.329967,330.499228,1898.913699,614.378917,357.971456,46


In [38]:
all_samples = df_intervention_age.index
all_samples

Index(['SRR17215646', 'SRR17215657', 'SRR17215658', 'SRR17215659',
       'SRR17215660', 'SRR17215661', 'SRR17215662', 'SRR17215663',
       'SRR17215664', 'SRR17215665', 'SRR17215666', 'SRR17215667',
       'SRR17215668', 'SRR17215669', 'SRR17215670', 'SRR17215671',
       'SRR17215672', 'SRR17215647', 'SRR17215648', 'SRR17215649',
       'SRR17215650', 'SRR17215651', 'SRR17215652', 'SRR17215653',
       'SRR17215654', 'SRR17215655', 'SRR17215656'],
      dtype='object')

In [39]:
#all not in select_WT_samples
test_samples = [s for s in all_samples if s not in select_WT_samples]
test_samples

['SRR17215657',
 'SRR17215658',
 'SRR17215659',
 'SRR17215660',
 'SRR17215661',
 'SRR17215662',
 'SRR17215663',
 'SRR17215664',
 'SRR17215665',
 'SRR17215666',
 'SRR17215667',
 'SRR17215668',
 'SRR17215669',
 'SRR17215670',
 'SRR17215671',
 'SRR17215672',
 'SRR17215648',
 'SRR17215649',
 'SRR17215650',
 'SRR17215651',
 'SRR17215652',
 'SRR17215655',
 'SRR17215656']

In [40]:
test_samples_filtered = test_samples
test_samples_filtered

['SRR17215657',
 'SRR17215658',
 'SRR17215659',
 'SRR17215660',
 'SRR17215661',
 'SRR17215662',
 'SRR17215663',
 'SRR17215664',
 'SRR17215665',
 'SRR17215666',
 'SRR17215667',
 'SRR17215668',
 'SRR17215669',
 'SRR17215670',
 'SRR17215671',
 'SRR17215672',
 'SRR17215648',
 'SRR17215649',
 'SRR17215650',
 'SRR17215651',
 'SRR17215652',
 'SRR17215655',
 'SRR17215656']

In [41]:
df_intervention_test = df_intervention_age.loc[
    df_intervention_age.index.isin(test_samples_filtered)
]

df_intervention_test.head(3)

Run,LOC107380275,LOC107389369,klf12,enpp2,LOC107382607,LOC107386405,wwox,LOC107388920,csgr14h3orf70,LOC107376005,...,LOC107373016,LOC107392495,LOC107396697,LOC107391667,tissue,age,sex,training,genotype,feeding_condition
SRR17215657,0.000000,0.0,485.228400,18.835113,40.360957,45.742418,43.051688,37.670227,47.536239,258.310128,...,0.0,163.237650,0.0,0.0,Liver,46,male,No,Het,full
SRR17215658,1.014282,0.0,454.398481,9.128541,54.771245,43.614140,50.714116,55.785528,51.728398,278.927639,...,0.0,146.056655,0.0,0.0,Liver,46,male,No,Het,full
SRR17215659,4.449614,0.0,422.713353,21.135668,54.507774,58.957389,54.507774,63.407003,87.879881,273.651276,...,0.0,151.286884,0.0,0.0,Liver,46,male,No,Het,full


In [42]:
df_intervention_test = df_intervention_test.drop(df_intervention_test.columns[[-1,-2,-3, -4, -6]], axis=1)
df_intervention_test.head(2)

Run,LOC107380275,LOC107389369,klf12,enpp2,LOC107382607,LOC107386405,wwox,LOC107388920,csgr14h3orf70,LOC107376005,...,LOC107394127,golph3,LOC107381980,LOC107390548,lhfpl5,LOC107373016,LOC107392495,LOC107396697,LOC107391667,age
SRR17215657,0.000000,0.0,485.228400,18.835113,40.360957,45.742418,43.051688,37.670227,47.536239,258.310128,...,0.0,254.722487,0.0,44.845508,0.0,0.0,163.237650,0.0,0.0,46
SRR17215658,1.014282,0.0,454.398481,9.128541,54.771245,43.614140,50.714116,55.785528,51.728398,278.927639,...,0.0,269.799098,0.0,29.414187,0.0,0.0,146.056655,0.0,0.0,46


In [43]:
#check cols ordered the same
df_intervention_test = df_intervention_test[atlas_plus_intervention.columns]

In [44]:
#subset your metadata
df_intervention_test_metadata = df_intervention_metadata.loc[
    df_intervention_metadata.index.isin(test_samples_filtered)
]

df_intervention_test_metadata.head(3)

,lib,age,Assay.Type,AvgSpotLen,Bases,BioProject,BioSample,Bytes,Center.Name,Consent,...,create_date,version,Sample.Name,sex,source_name,SRA.Study,tissue,file,UsedForTraining,ExcludedFromAnalysis
Run,,,,,,,,,,,,,,,,,,,,,
SRR17215657,SRR17215657,6.5 weeks,RNA-Seq,74,2840020025,PRJNA788399,SAMN23970512,1302718445,GEO,public,...,2021-12-13T12:07:00Z,1,GSM5730992,male,Nfr Liver,SRP350511,Liver,SRR17215657.featureCounts,No,Include
SRR17215658,SRR17215658,6.5 weeks,RNA-Seq,74,2919651799,PRJNA788399,SAMN23970491,1345051064,GEO,public,...,2021-12-13T12:07:00Z,1,GSM5730993,male,Nfr Liver,SRP350511,Liver,SRR17215658.featureCounts,No,Include
SRR17215659,SRR17215659,6.5 weeks,RNA-Seq,74,2295530551,PRJNA788399,SAMN23970492,1061958088,GEO,public,...,2021-12-13T12:04:00Z,1,GSM5730994,male,Nfr Liver,SRP350511,Liver,SRR17215659.featureCounts,No,Include


In [45]:
age = atlas_plus_intervention['age']
X = atlas_plus_intervention.iloc[:, :-1]

# Separate training and test data
X_train = X
age_train = age
X_test = df_intervention_test.iloc[:,:-1]

# Scale the training and test data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape)

# Initialize lists to store results
train_results = []

(35, 25122)


In [ ]:
# # Loop through all possible components to predict the test set
# max_components = min(X_train_scaled.shape[0], X_train_scaled.shape[1])
# all_y_test_preds = []

# for n_components in range(1, max_components + 1):
#     print(f"Components: {n_components}")

#     # Apply PCA
#     pca = PCA(n_components=n_components, random_state=1)
#     X_train_pca = pca.fit_transform(X_train_scaled)
#     X_test_pca = pca.transform(X_test_scaled)

#     # Train the model
#     regressor = LinearRegression()
#     regressor.fit(X_train_pca, age_train)

#     # Predict on the test set
#     y_test_pred = regressor.predict(X_test_pca)

#     # Save the actual and predicted values to a DataFrame
#     results_df = pd.DataFrame({
#     'y_pred': y_test_pred
#     }, index=df_intervention_test.index.tolist()) # this should be the test dataset

#     df_intervention = results_df.loc[df_intervention_test.index.tolist(), ]

#     df_intervention.head(5)

#     # Make sure index is treated as string
#     sample_names = df_intervention.index.to_series().astype(str)

#     print(sample_names)

#     # Extract treatment (between first and second period)
#     df_intervention["genotype"] = df_intervention_test_metadata["genotype"]
#     df_intervention["age"] = df_intervention_test_metadata["age"]
#     df_intervention["feeding_condition"] = df_intervention_test_metadata["feeding_condition"]
#     df_intervention["age_genotype_feeding"] = df_intervention["age"].astype(str) + "_" + df_intervention["genotype"].astype(str)+ "_" + df_intervention["feeding_condition"].astype(str)
#     df_intervention = df_intervention.sort_values(by="age_genotype_feeding")
#     print(df_intervention)

#     # Create the box plot
#     plt.figure(figsize=(10, 6))
#     box = sns.boxplot(x='age_genotype_feeding', y='y_pred', data=df_intervention, palette='deep', showfliers = False, whis = [0,100])

#     #custom_palette = ['#FF6347', '#4682B4', '#32CD32', '#FFD700']  # Use desired hex color codes
#     custom_palette = ['#273276', '#B14325']  # Use desired hex color codes

#     # Add data points with custom colors
#     strip = sns.stripplot(
#         x='age_genotype_feeding',
#         y='y_pred',
#         data=df_intervention,
#         palette=custom_palette,  # Custom color palette
#         dodge=False,
#         alpha=0.9,
#         jitter=True,
#         size=15
#     )

#     plt.xticks(rotation=45)  # Rotate x-axis labels
#     plt.show()

#     # all_y_test_preds.append({
#     #     'n_components': n_components,
#     #     'y_test_pred': y_test_pred
#     # })

#     # Optionally, print the predictions for each number of components
#     #print(f"Components: {n_components}, Predictions: {y_test_pred}")

#     # # Save the DataFrame to a CSV file
#     # output_path = './outputs/pc_regression/'
#     # pc_num = "".join([str(n_components), "pcs"])
#     # out_file_prefix = 'ALDR_male_PCregression_pred'
#     # out_file_name = "_".join([out_file_prefix, pc_num])
#     # df_intervention.to_csv(os.path.join(output_path, out_file_name), sep="\t")

##### Select PCA number, export to csv

In [46]:
n_components = 18

# Apply PCA
pca = PCA(n_components=n_components,svd_solver='full',random_state=1)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Train the model
regressor = LinearRegression()
regressor.fit(X_train_pca, age_train)

# Predict on the test set
y_test_pred = regressor.predict(X_test_pca)

# Save the actual and predicted values to a DataFrame
results_df = pd.DataFrame({
'y_pred': y_test_pred
}, index=df_intervention_test.index.tolist()) # this should be the test dataset

df_intervention = results_df.loc[df_intervention_test.index.tolist(), ]

df_intervention.head(5)

# Make sure index is treated as string
sample_names = df_intervention.index.to_series().astype(str)

print(sample_names)

# Extract treatment (between first and second period)
df_intervention["genotype"] = df_intervention_test_metadata["genotype"]
df_intervention["age"] = df_intervention_test_metadata["age"]
df_intervention["age_genotype"] = df_intervention["age"].astype(str) + "_" + df_intervention["genotype"].astype(str)
df_intervention["feeding_condition"] = df_intervention_test_metadata["feeding_condition"]
df_intervention["age_genotype_feeding"] = df_intervention["age"].astype(str) + "_" + df_intervention["genotype"].astype(str)+ "_" + df_intervention["feeding_condition"].astype(str)


print(df_intervention)
df_intervention.to_csv("./predictions/Astre_ALmaleliver_maleclock_ALLpredictions_18PCs_EC-250724.csv")
print("Solver actually used:", pca._fit_svd_solver)


SRR17215657    SRR17215657
SRR17215658    SRR17215658
SRR17215659    SRR17215659
SRR17215660    SRR17215660
SRR17215661    SRR17215661
SRR17215662    SRR17215662
SRR17215663    SRR17215663
SRR17215664    SRR17215664
SRR17215665    SRR17215665
SRR17215666    SRR17215666
SRR17215667    SRR17215667
SRR17215668    SRR17215668
SRR17215669    SRR17215669
SRR17215670    SRR17215670
SRR17215671    SRR17215671
SRR17215672    SRR17215672
SRR17215648    SRR17215648
SRR17215649    SRR17215649
SRR17215650    SRR17215650
SRR17215651    SRR17215651
SRR17215652    SRR17215652
SRR17215655    SRR17215655
SRR17215656    SRR17215656
dtype: object
                y_pred genotype        age   age_genotype feeding_condition  \
SRR17215657  76.524370      Het  6.5 weeks  6.5 weeks_Het              full   
SRR17215658  76.160952      Het  6.5 weeks  6.5 weeks_Het              full   
SRR17215659  73.301288      Het  6.5 weeks  6.5 weeks_Het              full   
SRR17215660  88.550299       WT   15 weeks    15 

# Rotate training samples

In [50]:
# --------------------------------------------------------------------
# Config
# --------------------------------------------------------------------
OUTDIR = "./predictions_exhaustive"
os.makedirs(OUTDIR, exist_ok=True)

# If you want to cap PCs to a smaller number (e.g., 100), set here; else leave None to use full allowable range
PC_CAP = 20   # e.g., PC_CAP = 100

# --------------------------------------------------------------------
# Sample selection setup (same as your BayesAge loop)
# --------------------------------------------------------------------
WT_samples = df_intervention_age[
    (df_intervention_age['genotype'] == 'WT') &
    (df_intervention_age['feeding_condition'] == 'full')
].index

young_samples = sorted([s for s in WT_samples if df_intervention_age.loc[s, 'age'] == 46])
old_samples   = sorted([s for s in WT_samples if df_intervention_age.loc[s, 'age'] == 105])

# All 2x2 combinations
young_combos = list(itertools.combinations(young_samples, 2))
old_combos   = list(itertools.combinations(old_samples, 2))


In [52]:
random.seed(123)
# --------------------------------------------------------------------
# Main loop: iterate over all combos, then sweep PC numbers
# --------------------------------------------------------------------
for y_combo in young_combos:
    for o_combo in old_combos:

        combo_id = f"{'_'.join(map(str, y_combo))}__{'_'.join(map(str, o_combo))}"
        print(f"\nRunning PCR for combo: {combo_id}")

        # ----------------------------
        # Build TRAIN/ATLAS combined
        # ----------------------------
        select_train_samples = list(y_combo) + list(o_combo)

        df_intervention_train_full = df_intervention_age.loc[select_train_samples]

        # Use tissue_df_age (counts + age + meta) for atlas, then append intervention rows
        atlas_plus_intervention = pd.concat([tissue_df_age, df_intervention_train_full], axis=0)

        # Age target (pull BEFORE dropping meta columns)
        y_train_age = atlas_plus_intervention['age'].astype(float)

        # Training features: drop metadata columns by your index pattern
        X_train_counts = atlas_plus_intervention.drop(
            atlas_plus_intervention.columns[[-1, -2, -3, -4, -5, -6]], axis=1
        )

        # ----------------------------
        # Build TEST matrices/metadata
        # ----------------------------
        test_samples = [s for s in df_intervention_age.index if s not in select_train_samples]
        df_intervention_test_full = df_intervention_age.loc[test_samples]

        # Drop the same metadata columns for TEST and align to train columns
        X_test_counts = df_intervention_test_full.drop(
            df_intervention_test_full.columns[[-1, -2, -3, -4,-5, -6]], axis=1
        )
        X_test_counts = X_test_counts[X_train_counts.columns]

        # Test metadata for annotations
        test_meta = df_intervention_age.loc[
            test_samples, ['genotype', 'age', 'feeding_condition']
        ].copy()
        test_meta['age_genotype_feeding'] = (
            test_meta['age'].astype(str) + "_" +
            test_meta['genotype'].astype(str) + "_" +
            test_meta['feeding_condition'].astype(str)
        )

        # ----------------------------
        # Scale and PCR
        # ----------------------------
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_counts.values)
        X_test_scaled  = scaler.transform(X_test_counts.values)

        # Determine allowable PCs
        max_components_allowed = min(X_train_scaled.shape[0], X_train_scaled.shape[1])
        if PC_CAP is not None:
            max_components_allowed = min(max_components_allowed, PC_CAP)

        for n_components in range(1, max_components_allowed + 1):
            print(f"  PCs: {n_components}")

            # PCA on training, transform both
            pca = PCA(n_components=n_components,svd_solver = 'full', random_state=1)
            X_train_pca = pca.fit_transform(X_train_scaled)
            X_test_pca  = pca.transform(X_test_scaled)

            # Linear regression on PCs → age
            regressor = LinearRegression()
            regressor.fit(X_train_pca, y_train_age.values)

            # Predict for test samples
            y_test_pred = regressor.predict(X_test_pca)

            # Package output with metadata
            df_pred = pd.DataFrame({'y_pred': y_test_pred}, index=test_samples)
            df_pred = df_pred.join(test_meta, how='left').sort_values(by="age_genotype_feeding")

            # Save
            out_prefix = f"PCR_pred_{combo_id}_pc{n_components:03d}"
            out_path = os.path.join(OUTDIR, f"{out_prefix}.tsv")
            df_pred.to_csv(out_path, sep="\t")



Running PCR for combo: SRR17215653_SRR17215654__SRR17215646_SRR17215647
  PCs: 1
  PCs: 2
  PCs: 3
  PCs: 4
  PCs: 5
  PCs: 6
  PCs: 7
  PCs: 8
  PCs: 9
  PCs: 10
  PCs: 11
  PCs: 12
  PCs: 13
  PCs: 14
  PCs: 15
  PCs: 16
  PCs: 17
  PCs: 18
  PCs: 19
  PCs: 20

Running PCR for combo: SRR17215653_SRR17215654__SRR17215646_SRR17215648
  PCs: 1
  PCs: 2
  PCs: 3
  PCs: 4
  PCs: 5
  PCs: 6
  PCs: 7
  PCs: 8
  PCs: 9
  PCs: 10
  PCs: 11
  PCs: 12
  PCs: 13
  PCs: 14
  PCs: 15
  PCs: 16
  PCs: 17
  PCs: 18
  PCs: 19
  PCs: 20

Running PCR for combo: SRR17215653_SRR17215654__SRR17215647_SRR17215648
  PCs: 1
  PCs: 2
  PCs: 3
  PCs: 4
  PCs: 5
  PCs: 6
  PCs: 7
  PCs: 8
  PCs: 9
  PCs: 10
  PCs: 11
  PCs: 12
  PCs: 13
  PCs: 14
  PCs: 15
  PCs: 16
  PCs: 17
  PCs: 18
  PCs: 19
  PCs: 20

Running PCR for combo: SRR17215653_SRR17215655__SRR17215646_SRR17215647
  PCs: 1
  PCs: 2
  PCs: 3
  PCs: 4
  PCs: 5
  PCs: 6
  PCs: 7
  PCs: 8
  PCs: 9
  PCs: 10
  PCs: 11
  PCs: 12
  PCs: 13
  PCs: 14
  PC